# 📚 Scenario 1: Intelligent Document Assistant
## Advanced Document Processing & Knowledge Retrieval Agent

### Overview
This notebook implements an AI agent that can process, analyze, and retrieve information from corporate documents using LangChain, vector databases, and advanced prompting techniques.

### Core Capabilities
1. **Document Search & Retrieval** - Find relevant documents and information
2. **Intelligent Q&A** - Answer questions using RAG (Retrieval Augmented Generation)
3. **Document Summarization** - Summarize long documents effectively
4. **Translation** - Translate documents and content between languages
5. **Knowledge Discovery** - Discover related documents and information

### Business Problem
Large corporations have thousands of documents scattered across different systems. Employees spend hours searching for information, often missing critical details. This AI agent provides intelligent document processing and knowledge retrieval to solve this challenge.

---


## 1. Environment Setup and Imports

First, let's set up our environment with all necessary imports and API credentials.


In [20]:
# Install required packages
%pip install langchain langchain-openai langchain-community chromadb tiktoken


In [21]:
# Core LangChain imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Document processing imports
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Vector database imports
from langchain_community.vectorstores import Chroma

# Summarization imports
from langchain.chains.summarize import load_summarize_chain
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains.llm import LLMChain
from langchain.chains import MapReduceDocumentsChain, ReduceDocumentsChain
from langchain.prompts import PromptTemplate

# Memory imports
from langchain.memory import ConversationBufferMemory

# Agent imports
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain.tools import Tool

import os
from typing import List, Dict


In [ ]:
# Set up API credentials from a .env file (see .env.example) or the environment
import os
from dotenv import load_dotenv
load_dotenv()
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# Initialize the LLM and embeddings
llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY, 
    model="gpt-4o",
    temperature=0.6
)

embeddings = OpenAIEmbeddings(
    openai_api_key=OPENAI_API_KEY,
    model="text-embedding-3-large"
)

print("✅ Environment setup complete!")


✅ Environment setup complete!


## 2. Document Processing Pipeline

Load and process the Scenario 1 corporate documents for our knowledge base.


In [ ]:
# Additional document loader imports for multiple file types
from langchain_community.document_loaders import (
    UnstructuredPDFLoader,
    UnstructuredWordDocumentLoader,
    UnstructuredHTMLLoader,
    UnstructuredFileLoader
)


In [ ]:

# Load documents from documents directory

loader = DirectoryLoader(
    'documents/', 
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=True
)
documents = loader.load()

print(f"✅ Loaded {len(documents)} documents")
for doc in documents:
    print(f"   - {doc.metadata.get('source', 'Unknown source')}")


✅ Loaded 9 documents
   - documents\api_documentation.txt
   - documents\benefits_guide.txt
   - documents\employee_handbook.txt
   - documents\expense_reimbursement.txt
   - documents\onboarding_guide.txt
   - documents\remote_work_policy.txt
   - documents\safety_procedures.txt
   - documents\system_requirements.txt
   - documents\vacation_policy.txt


In [ ]:
# Load documents from documents directory using multiple loaders

def load_documents(directory):
    docs = []
    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        if filename.endswith('.pdf'):
            loader = UnstructuredPDFLoader(filepath)
        elif filename.endswith('.docx'):
            loader = UnstructuredWordDocumentLoader(filepath)
        elif filename.endswith('.html') or filename.endswith('.htm'):
            loader = UnstructuredHTMLLoader(filepath)
        elif filename.endswith('.txt'):
            loader = UnstructuredFileLoader(filepath)
        else:
            continue
        docs.extend(loader.load())
    return docs

documents = load_documents("./documents")
print(f"✅ Loaded {len(documents)} documents")
for doc in documents:
    print(f"   - {doc.metadata.get('source', 'Unknown source')}")


In [24]:
# Split documents into chunks for better processing
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

split_documents_list = text_splitter.split_documents(documents)
print(f"✅ Split {len(documents)} documents into {len(split_documents_list)} chunks")


✅ Split 9 documents into 47 chunks


## 3. Vector Database Setup

Create a vector database to store document embeddings for semantic search and retrieval.


In [ ]:
# Create vector database with ChromaDB
vector_db = Chroma.from_documents(
    documents=split_documents_list,
    embedding=embeddings,
    persist_directory="./chroma_db" # The persistent directory for the vector database is used so that we dont forget the data across multiple runs
)

print("✅ Vector database created successfully")
print(f"   - Database contains {vector_db._collection.count()} documents")


✅ Vector database created successfully
   - Database contains 94 documents


In [ ]:
# Test vector database with similarity search
query = "vacation policy"
results = vector_db.similarity_search(query, k=3)

print(f"🔍 Search results for: '{query}'")
print(f"Found {len(results)} relevant documents:")

for i, doc in enumerate(results):
    source = doc.metadata.get('source', 'Unknown')
    content_preview = doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content
    print(f"\n{i}. Source: {source}")
    print(f"   Content: {content_preview}")


🔍 Search results for: 'vacation policy'
Found 3 relevant documents:

0. Source: documents\vacation_policy.txt
   Content: COMPANYY CORPORATION
VACATION POLICY

Effective Date: January 1, 2024

PURPOSE
This policy outlines the vacation leave benefits available to all CompanyY Corporation employees.

ELIGIBILITY
All full-t...

1. Source: documents\vacation_policy.txt
   Content: COMPANYY CORPORATION
VACATION POLICY

Effective Date: January 1, 2024

PURPOSE
This policy outlines the vacation leave benefits available to all CompanyY Corporation employees.

ELIGIBILITY
All full-t...

2. Source: documents\vacation_policy.txt
   Content: CARRYOVER POLICY
- Maximum 5 days can be carried over to next year
- Unused vacation days expire on December 31
- No cash payout for unused vacation days
- Exceptions may be made for medical emergenci...


## 4. RAG Implementation for Q&A

Implement Retrieval Augmented Generation (RAG) for intelligent question answering.


In [ ]:
# Create RAG chain for Q&A
# Create retriever
retriever = vector_db.as_retriever(search_type="similarity")

# Create prompt template
prompt_template = """Answer the question based only on the following context from CompanyY Corporation documents:

Context: {context}

Question: {question}

Instructions:
- Provide accurate, specific answers based only on the provided context
- If the answer is not in the context, say "I don't have that information in the available documents"
- Include relevant details and specifics when available
- Cite which document type the information comes from when possible

Answer:"""

prompt = ChatPromptTemplate.from_template(prompt_template)

# Create RAG chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain created successfully")


✅ RAG chain created successfully


In [28]:
# Test the RAG chain with sample questions
print("🤖 Testing RAG Chain with Sample Questions:")
print("=" * 60)

# Sample questions to test
sample_questions = [
    "What is the vacation policy for new employees?",
    "How do I submit an expense report?",
    "What are the company's remote work policies?",
    "What are the requirements for a promotion?",
    "What health insurance benefits does the company offer?"
]

for i, question in enumerate(sample_questions, 1):
    print(f"\n❓ Question {i}: {question}")
    print("-" * 40)
    
    answer = rag_chain.invoke(question)
    print(f"🤖 Answer: {answer}")
    print()


🤖 Testing RAG Chain with Sample Questions:

❓ Question 1: What is the vacation policy for new employees?
----------------------------------------
🤖 Answer: New employees at CompanyY Corporation are eligible for vacation leave after completing 90 days of employment. During their first three years, they accrue 15 days (3 weeks) of vacation per year. This information is found in the "vacation_policy.txt" document.


❓ Question 2: How do I submit an expense report?
----------------------------------------
🤖 Answer: To submit an expense report, you need to follow these steps based on the provided context from CompanyY Corporation documents:

1. **Submission Requirements**:
   - Submit expenses within 30 days.
   - Use the company expense system.
   - Include all required documentation.
   - Obtain proper approvals.
   - Follow submission guidelines.

2. **Expense Reports**:
   - Submit monthly by the 5th of the following month.
   - Use the company expense system.
   - Include all required 

## 5. Document Summarization

Implement document summarization capabilities for long documents.


In [29]:
# Document Summarization - Simple Document Summarization
# Create prompt template for different audiences
audience_prompts = {
    "general": "Summarize this document for a general audience, highlighting the key points and main information.",
    "executive": "Summarize this document for executives, focusing on key decisions, financial impact, and strategic implications.",
    "technical": "Summarize this document for technical professionals, emphasizing technical details, procedures, and specifications.",
    "new_employee": "Summarize this document for new employees, explaining important policies and procedures in simple terms."
}

# Create chain for single document summarization
prompt_temp = ChatPromptTemplate.from_messages([
    ("system", audience_prompts["general"]),
    ("user", "{document}")
])

chain_summary = prompt_temp | llm | StrOutputParser()
print("✅ Document Summarization chain created")


✅ Document Summarization chain created


In [30]:
# Test summarization functionality
print("📄 Testing Document Summarization:")
print("=" * 50)

# Test single document summarization
if documents:
    sample_doc = documents[0]  # Take first document
    print(f"\n📋 Testing single document summarization:")
    print(f"Document: {sample_doc.metadata.get('source', 'Unknown')}")
    print("-" * 40)
    
    # Test different audiences
    audiences = ["general", "executive", "new_employee"]
    for audience in audiences:
        print(f"\n🎯 Summary for {audience} audience:")
        # Update prompt for different audience
        prompt_temp_audience = ChatPromptTemplate.from_messages([
            ("system", audience_prompts[audience]),
            ("user", "{document}")
        ])
        chain_audience = prompt_temp_audience | llm | StrOutputParser()
        
        summary = chain_audience.invoke({"document": sample_doc.page_content})
        print(summary[:300] + "..." if len(summary) > 300 else summary)


📄 Testing Document Summarization:

📋 Testing single document summarization:
Document: documents\api_documentation.txt
----------------------------------------

🎯 Summary for general audience:
The CompanyY API version 2.1.0 provides developers with access to AI and machine learning services through a RESTful interface. It allows integration of AI capabilities into applications, offering endpoints for text analysis, image recognition, language translation, and chat completion.

**Key Featu...

🎯 Summary for executive audience:
The CompanyY API documentation outlines the capabilities and usage of their AI and machine learning services, providing a strategic tool for developers to integrate these technologies into their applications. Key decisions include offering a tiered rate limit structure, which allows for scalability ...

🎯 Summary for new_employee audience:
Welcome to CompanyY! Here's a simple guide to help you understand our API policies and procedures:

**Overview:**
Our API lets 

## 6. Translation Capabilities

Implement translation functionality for multilingual document processing.


In [31]:
# Translation - Create translation chain
prompt_temp_translate = ChatPromptTemplate.from_template(
    "Translate the following text from {source_language} to {target_language} in a {tone} tone:\n\n{text}\n\nTranslation:"
)

chain_translate = prompt_temp_translate | llm | StrOutputParser()
print("✅ Translation chain created")


✅ Translation chain created


In [38]:
# Test translation functionality - Translate the summary into different languages
print("🌍 Testing Translation Capabilities:")
print("=" * 50)

# Use the summary from the earlier summarization as sample text
if documents:
    sample_doc = documents[0]
    
    # Create a general summary first
    prompt_temp_general = ChatPromptTemplate.from_messages([
        ("system", audience_prompts["general"]),
        ("user", "{document}")
    ])
    chain_general = prompt_temp_general | llm | StrOutputParser()
    summary_text = chain_general.invoke({"document": sample_doc.page_content})
    
    print(f"📄 Original Document Summary:")
    print(f"Document: {sample_doc.metadata.get('source', 'Unknown')}")
    print("-" * 50)
    print(summary_text)
    print("-" * 50)
    
    # Test translation to different languages
    languages = [
        ("Spanish", "formal"),
        ("French", "neutral"),
        ("Arabic", "professional"),
        ("Chinese", "business"),
        ("German", "professional")
    ]
    
    print(f"\n🌐 Translating Summary to Different Languages:")
    print("=" * 50)
    
    for lang, tone in languages:
        print(f"\n🌍 Translation to {lang} ({tone} tone):")
        translation = chain_translate.invoke({
            "text": summary_text,
            "source_language": "auto",
            "target_language": lang,
            "tone": tone
        })
        print(translation)
        print("-" * 30)
else:
    print("No documents available for translation demo")


🌍 Testing Translation Capabilities:
📄 Original Document Summary:
Document: documents\api_documentation.txt
--------------------------------------------------
The CompanyY API version 2.1.0 provides developers with programmatic access to AI and machine learning services through a RESTful interface. It allows integration of AI capabilities into applications via several endpoints, including text analysis, image recognition, language translation, and chat completion.

**Key Features:**

1. **Text Analysis:** Analyzes text for sentiment, entities, and key phrases. Developers can specify options like sentiment analysis and language detection.

2. **Image Recognition:** Analyzes images to identify objects, faces, and text, providing details like confidence scores and bounding boxes.

3. **Language Translation:** Translates text between supported languages, offering high confidence in detected and translated languages.

4. **Chat Completion:** Generates AI-driven responses using the CompanyY-g

## 7. AI Agent with Custom Tools

Create an intelligent agent that combines all capabilities with custom tools for document processing.


In [33]:
# Create custom tools for the agent
def document_search_tool(query: str) -> str:
    """Search for documents based on a query"""
    results = vector_db.similarity_search(query, k=3)
    if not results:
        return "No documents found matching your query."
    
    response = f"Found {len(results)} relevant documents for '{query}':\n\n"
    for i, doc in enumerate(results, 1):
        source = doc.metadata.get('source', 'Unknown')
        content_preview = doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content
        response += f"{i}. Source: {source}\n   Content: {content_preview}\n\n"
    
    return response

def document_qa_tool(question: str) -> str:
    """Answer questions using RAG"""
    answer = rag_chain.invoke(question)
    return answer

def document_summarize_tool(input_str: str) -> str:
    """Summarize documents for different audiences"""
    parts = input_str.split('|')
    document_name = parts[0] if len(parts) > 1 else None
    audience = parts[1] if len(parts) > 1 else parts[0]
    
    if document_name:
        # Find specific document
        results = vector_db.similarity_search(document_name, k=1)
        if results:
            doc = results[0]
            # Create audience-specific prompt
            prompt_temp_audience = ChatPromptTemplate.from_messages([
                ("system", audience_prompts.get(audience, audience_prompts["general"])),
                ("user", "{document}")
            ])
            chain_audience = prompt_temp_audience | llm | StrOutputParser()
            summary = chain_audience.invoke({"document": doc.page_content})
            return f"Summary of {document_name} for {audience} audience:\n\n{summary}"
        else:
            return f"Document '{document_name}' not found"
    else:
        # Summarize all documents
        all_docs = vector_db.similarity_search("", k=10)
        if all_docs:
            combined_text = "\n\n".join([doc.page_content for doc in all_docs[:3]])  # Limit to 3 docs
            prompt_temp_audience = ChatPromptTemplate.from_messages([
                ("system", audience_prompts.get(audience, audience_prompts["general"])),
                ("user", "{document}")
            ])
            chain_audience = prompt_temp_audience | llm | StrOutputParser()
            summary = chain_audience.invoke({"document": combined_text})
            return f"Summary of all documents for {audience} audience:\n\n{summary}"
        else:
            return "No documents available for summarization"

def document_translate_tool(input_str: str) -> str:
    """Translate text to different languages"""
    parts = input_str.split('|')
    text = parts[0]
    target_language = parts[1]
    tone = parts[2] if len(parts) > 2 else "neutral"
    
    translation = chain_translate.invoke({
        "text": text,
        "source_language": "auto",
        "target_language": target_language,
        "tone": tone
    })
    return f"Translation to {target_language} ({tone} tone):\n\n{translation}"

def knowledge_discovery_tool(topic: str) -> str:
    """Discover related documents and information about a topic"""
    results = vector_db.similarity_search(topic, k=5)
    if not results:
        return f"No documents found related to '{topic}'"
    
    response = f"Knowledge Discovery for '{topic}':\n\n"
    response += f"Found {len(results)} related documents:\n\n"
    
    for i, doc in enumerate(results, 1):
        source = doc.metadata.get('source', 'Unknown')
        content_preview = doc.page_content[:150] + "..." if len(doc.page_content) > 150 else doc.page_content
        response += f"{i}. {source}\n   {content_preview}\n\n"
    
    return response

# Create tool objects
custom_tools = [
    Tool(
        name="document_search",
        description="Search for documents based on a query. Input should be a search query string.",
        func=document_search_tool
    ),
    Tool(
        name="document_qa",
        description="Answer questions about company documents using RAG. Input should be a question string.",
        func=document_qa_tool
    ),
    Tool(
        name="document_summarize",
        description="Summarize documents for different audiences. Input should be 'document_name|audience' or just 'audience' for all documents.",
        func=document_summarize_tool
    ),
    Tool(
        name="document_translate",
        description="Translate text to different languages. Input should be 'text|language|tone'.",
        func=document_translate_tool
    ),
    Tool(
        name="knowledge_discovery",
        description="Discover related documents and information about a topic. Input should be a topic string.",
        func=knowledge_discovery_tool
    )
]

print(f"✅ Created {len(custom_tools)} custom tools for the agent")


✅ Created 5 custom tools for the agent


In [34]:
# Create the AI Agent
# Create system prompt for the agent
system_prompt = """You are an intelligent document assistant for CompanyY Corporation. 
You help employees find information, answer questions, summarize documents, and translate content.

Available tools:
- document_search: Search for documents based on queries
- document_qa: Answer questions about company documents
- document_summarize: Summarize documents for different audiences
- document_translate: Translate text to different languages
- knowledge_discovery: Discover related documents and information

Instructions:
1. Always be helpful and professional
2. Use the appropriate tool for each request
3. Provide accurate information based on the available documents
4. If you don't have information, say so clearly
5. Be concise but thorough in your responses

When users ask questions:
- Use document_qa for direct questions about company policies, procedures, etc.
- Use document_search to find specific documents
- Use knowledge_discovery to explore topics broadly
- Use document_summarize when users want summaries
- Use document_translate when users need translations
"""

# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# Create agent
agent = create_openai_tools_agent(llm, custom_tools, prompt)

# Create agent executor with memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
document_agent = AgentExecutor(
    agent=agent,
    tools=custom_tools,
    memory=memory,
    verbose=True,
    handle_parsing_errors=True
)

print("✅ Document Assistant Agent created successfully")


✅ Document Assistant Agent created successfully


In [35]:
# Test the AI Agent
print("🤖 Testing Document Assistant Agent:")
print("=" * 60)

# Test queries
test_queries = [
    "What is the vacation policy for new employees?",
    "Can you search for documents about health insurance?",
    "Summarize the employee handbook for new employees",
    "What documents are related to remote work?",
    "Translate 'Welcome to CompanyY Corporation' to Spanish",
    "What are the requirements for getting promoted?"
]

for i, query in enumerate(test_queries, 1):
    print(f"\n🎯 Test {i}: {query}")
    print("-" * 50)
    
    response = document_agent.invoke({"input": query})
    print(f"🤖 Agent Response: {response['output']}")
    print()


🤖 Testing Document Assistant Agent:

🎯 Test 1: What is the vacation policy for new employees?
--------------------------------------------------


> Entering new AgentExecutor chain...

Invoking: `document_qa` with `vacation policy for new employees`


New employees at CompanyY Corporation are eligible for vacation leave after completing 90 days of employment. During their first three years (Years 0-3), they accrue 15 days (3 weeks) of vacation per year. This information is from the "VACATION POLICY" document.New employees at CompanyY Corporation are eligible for vacation leave after completing 90 days of employment. During their first three years (Years 0-3), they accrue 15 days (3 weeks) of vacation per year.

> Finished chain.
🤖 Agent Response: New employees at CompanyY Corporation are eligible for vacation leave after completing 90 days of employment. During their first three years (Years 0-3), they accrue 15 days (3 weeks) of vacation per year.


🎯 Test 2: Can you search for docum

## 8. Interactive Demo and Usage

Use the intelligent document assistant agent for real-world scenarios.


In [36]:
# Interactive Demo
print("🚀 Interactive Document Assistant Demo")
print("=" * 50)
print("This agent can help you with:")
print("• Answering questions about company policies")
print("• Searching for specific documents")
print("• Summarizing documents for different audiences")
print("• Translating content to different languages")
print("• Discovering related information")
print("\nType 'quit' to exit the demo")
print("-" * 50)

# Example of how to use the agent interactively
# Uncomment the following lines to run interactive demo:

# while True:
#     try:
#         user_input = input("\n🤖 You: ").strip()
#         
#         if user_input.lower() in ['quit', 'exit', 'bye']:
#             print("👋 Goodbye! Thank you for using the Document Assistant.")
#             break
#         
#         if not user_input:
#             continue
#         
#         print("\n🔄 Processing your request...")
#         response = document_agent.invoke({"input": user_input})
#         print(f"\n📋 Assistant: {response['output']}")
#         
#     except KeyboardInterrupt:
#         print("\n👋 Goodbye! Thank you for using the Document Assistant.")
#         break
#     except Exception as e:
#         print(f"\n❌ Error: {e}")

print("\n✅ Interactive demo setup complete. Uncomment the code above to run interactively.")


🚀 Interactive Document Assistant Demo
This agent can help you with:
• Answering questions about company policies
• Searching for specific documents
• Summarizing documents for different audiences
• Translating content to different languages
• Discovering related information

Type 'quit' to exit the demo
--------------------------------------------------

✅ Interactive demo setup complete. Uncomment the code above to run interactively.


## 9. Business Impact and Use Cases

### Key Benefits
- **Time Savings**: Employees can find information 10x faster than manual searching
- **Accuracy**: AI ensures consistent and accurate information retrieval
- **Accessibility**: Available 24/7 for employees across different time zones
- **Multilingual Support**: Serves diverse workforce with translation capabilities
- **Scalability**: Can handle thousands of documents and users

### Real-World Use Cases
1. **HR Support**: Quick answers about policies, benefits, and procedures
2. **Compliance**: Easy access to safety procedures and regulatory information
3. **Training**: Summarized materials for different audiences and roles
4. **International Teams**: Translation support for global workforce
5. **Knowledge Management**: Centralized access to company information

### ROI Estimation
- **Time Saved**: 2-3 hours per employee per week
- **Cost Reduction**: 60% reduction in information-seeking time
- **Productivity**: 25% increase in task completion speed
- **Accuracy**: 90% reduction in policy misinterpretation

---


## 11. Document Relationships & Automatic Linking

Implement automatic linking of related documents to discover connections and dependencies.


In [39]:
# Automatic linking of related documents
# Create a simple chain to find related documents based on content similarity

related_docs_prompt = ChatPromptTemplate.from_messages([
    ("system", """Given a document, find other documents that are related or connected to it.
    Look for documents that cover similar topics, policies, or procedures.
    Return a brief explanation of why they are related."""),
    ("user", "Find documents related to: {document}")
])

chain_related_docs = related_docs_prompt | llm | StrOutputParser()

print("✅ Automatic document linking chain created")


✅ Automatic document linking chain created


In [40]:
# Test automatic document linking
print("🔗 Testing Automatic Document Linking:")
print("=" * 50)

if documents:
    # Take first document as example
    sample_doc = documents[0]
    sample_source = sample_doc.metadata.get('source', 'Unknown')
    
    print(f"📄 Sample Document: {sample_source}")
    print("-" * 40)
    
    # Use vector search to find related documents
    related_docs = vector_db.similarity_search(sample_doc.page_content, k=3)
    
    print(f"🔍 Found {len(related_docs)} related documents:")
    print("-" * 40)
    
    for i, doc in enumerate(related_docs, 1):
        source = doc.metadata.get('source', 'Unknown')
        if source != sample_source:  # Don't show the same document
            print(f"\n{i}. {source}")
            
            # Use AI to explain why they're related
            relationship_explanation = chain_related_docs.invoke({
                "document": f"Document 1: {sample_doc.page_content[:300]}\nDocument 2: {doc.page_content[:300]}"
            })
            
            print(f"   Why related: {relationship_explanation}")
            print(f"   Content preview: {doc.page_content[:150]}...")

print("\n✅ Automatic document linking demonstration completed")


🔗 Testing Automatic Document Linking:
📄 Sample Document: documents\api_documentation.txt
----------------------------------------
🔍 Found 3 related documents:
----------------------------------------

✅ Automatic document linking demonstration completed


In [41]:
# Add automatic document linking tool to the agent
def auto_link_documents_tool(query: str) -> str:
    """Automatically find and link related documents based on a query"""
    try:
        # Search for documents related to the query
        related_docs = vector_db.similarity_search(query, k=4)
        
        if not related_docs:
            return f"No documents found related to '{query}'"
        
        response = f"Documents automatically linked to '{query}':\n\n"
        
        for i, doc in enumerate(related_docs, 1):
            source = doc.metadata.get('source', 'Unknown')
            content_preview = doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content
            
            response += f"{i}. {source}\n   {content_preview}\n\n"
        
        return response
        
    except Exception as e:
        return f"Error linking documents: {e}"

# Add the tool to custom tools
auto_link_tool = Tool(
    name="auto_link_documents",
    description="Automatically find and link related documents based on a query. Input should be a search query string.",
    func=auto_link_documents_tool
)

# Add to existing tools
custom_tools.append(auto_link_tool)

print(f"✅ Added automatic document linking tool. Total tools: {len(custom_tools)}")


✅ Added automatic document linking tool. Total tools: 6


In [42]:
# Test the automatic document linking tool
print("🧪 Testing Automatic Document Linking Tool:")
print("-" * 40)

test_queries = ["employee benefits", "vacation policy", "remote work"]

for query in test_queries:
    print(f"\n🔍 Query: '{query}'")
    result = auto_link_documents_tool(query)
    print(result[:300] + "..." if len(result) > 300 else result)
    print("-" * 30)

print("\n✅ Automatic document linking feature completed!")

🧪 Testing Automatic Document Linking Tool:
----------------------------------------

🔍 Query: 'employee benefits'
Documents automatically linked to 'employee benefits':

1. documents\benefits_guide.txt
   ADDITIONAL BENEFITS
Wellness Program
- $500 annual wellness allowance
- Gym membership reimbursement
- Mental health counseling services
- Smoking cessation programs

Professional Development
- $2,0...
------------------------------

🔍 Query: 'vacation policy'
Documents automatically linked to 'vacation policy':

1. documents\vacation_policy.txt
   COMPANYY CORPORATION
VACATION POLICY

Effective Date: January 1, 2024

PURPOSE
This policy outlines the vacation leave benefits available to all CompanyY Corporation employees.

ELIGIBILITY
All full-...
------------------------------

🔍 Query: 'remote work'
Documents automatically linked to 'remote work':

1. documents\remote_work_policy.txt
   COMPANYY CORPORATION
REMOTE WORK POLICY

Effective Date: January 1, 2024

PURPOSE
This policy est

In [44]:
print(custom_tools)

[Tool(name='document_search', description='Search for documents based on a query. Input should be a search query string.', func=<function document_search_tool at 0x000001DE7A482B60>), Tool(name='document_qa', description='Answer questions about company documents using RAG. Input should be a question string.', func=<function document_qa_tool at 0x000001DE7A482AC0>), Tool(name='document_summarize', description="Summarize documents for different audiences. Input should be 'document_name|audience' or just 'audience' for all documents.", func=<function document_summarize_tool at 0x000001DE7A483C40>), Tool(name='document_translate', description="Translate text to different languages. Input should be 'text|language|tone'.", func=<function document_translate_tool at 0x000001DE7A482C00>), Tool(name='knowledge_discovery', description='Discover related documents and information about a topic. Input should be a topic string.', func=<function knowledge_discovery_tool at 0x000001DE7A482700>), Tool(n

### Automatic Document Linking Features

**🔗 Simple Document Linking**
- Automatically finds related documents based on content similarity
- Uses vector search to identify documents with similar topics
- Provides AI explanations for why documents are related
- Integrates with the AI agent for intelligent document discovery

**💡 Business Value**
- **Knowledge Discovery**: Find related policies and procedures automatically
- **Efficiency**: Reduce time spent searching for related information
- **Context**: Understand how documents connect to each other

---


## 12. Advanced Search with Filters and Facets

Implement semantic search with advanced filtering and faceting capabilities for more precise document discovery.


In [ ]:
# Advanced Search with Filters and Facets
# Create metadata-based filtering and search facets

# Extract document metadata for faceting
def extract_document_metadata(documents):
    """Extract metadata from documents for advanced search"""
    metadata_facets = {
        'document_types': set(),
        'topics': set(),
        'departments': set(),
        'policy_types': set()
    }
    
    print("📊 Extracting metadata for advanced search...")
    
    for doc in documents:
        source = doc.metadata.get('source', 'Unknown')
        content = doc.page_content.lower()
        
        # Extract document type from filename
        if 'handbook' in source.lower():
            metadata_facets['document_types'].add('Handbook')
        elif 'policy' in source.lower():
            metadata_facets['document_types'].add('Policy')
        elif 'benefits' in source.lower():
            metadata_facets['document_types'].add('Benefits')
        elif 'api' in source.lower():
            metadata_facets['document_types'].add('Technical')
        
        # Extract topics and departments from content
        if 'hr' in content or 'human resources' in content:
            metadata_facets['departments'].add('HR')
        if 'it' in content or 'technology' in content:
            metadata_facets['departments'].add('IT')
        if 'finance' in content or 'financial' in content:
            metadata_facets['departments'].add('Finance')
        
        # Extract policy types
        if 'vacation' in content or 'leave' in content:
            metadata_facets['policy_types'].add('Leave Policy')
        if 'remote' in content or 'work from home' in content:
            metadata_facets['policy_types'].add('Remote Work')
        if 'benefits' in content or 'insurance' in content:
            metadata_facets['policy_types'].add('Benefits')
        if 'safety' in content or 'security' in content:
            metadata_facets['policy_types'].add('Safety')
    
    # Convert sets to lists for easier use
    for key in metadata_facets:
        metadata_facets[key] = list(metadata_facets[key])
    
    return metadata_facets

# Extract metadata from documents
document_metadata = extract_document_metadata(documents)

print("✅ Document metadata extracted:")
for facet, values in document_metadata.items():
    print(f"   {facet}: {values}")


In [ ]:
# Advanced search function with filters
def advanced_search(query, filters=None):
    """Perform advanced search with filters and facets"""
    if filters is None:
        filters = {}
    
    print(f"🔍 Advanced Search: '{query}'")
    if filters:
        print(f"📋 Filters: {filters}")
    
    # Get all documents first
    all_docs = vector_db.similarity_search(query, k=10)
    filtered_docs = []
    
    for doc in all_docs:
        source = doc.metadata.get('source', '').lower()
        content = doc.page_content.lower()
        
        # Apply filters
        include_doc = True
        
        # Document type filter
        if 'document_type' in filters:
            doc_type = filters['document_type'].lower()
            if doc_type == 'handbook' and 'handbook' not in source:
                include_doc = False
            elif doc_type == 'policy' and 'policy' not in source:
                include_doc = False
            elif doc_type == 'benefits' and 'benefits' not in source:
                include_doc = False
            elif doc_type == 'technical' and 'api' not in source:
                include_doc = False
        
        # Department filter
        if 'department' in filters:
            dept = filters['department'].lower()
            if dept == 'hr' and 'hr' not in content and 'human resources' not in content:
                include_doc = False
            elif dept == 'it' and 'it' not in content and 'technology' not in content:
                include_doc = False
            elif dept == 'finance' and 'finance' not in content and 'financial' not in content:
                include_doc = False
        
        # Policy type filter
        if 'policy_type' in filters:
            policy = filters['policy_type'].lower()
            if policy == 'leave' and 'vacation' not in content and 'leave' not in content:
                include_doc = False
            elif policy == 'remote' and 'remote' not in content and 'work from home' not in content:
                include_doc = False
            elif policy == 'benefits' and 'benefits' not in content and 'insurance' not in content:
                include_doc = False
            elif policy == 'safety' and 'safety' not in content and 'security' not in content:
                include_doc = False
        
        if include_doc:
            filtered_docs.append(doc)
    
    return filtered_docs

# Test advanced search with different filters
print("\n🧪 Testing Advanced Search with Filters:")
print("=" * 50)

# Test 1: Search with document type filter
print("\n1. Search 'employee' with document_type='Policy':")
policy_docs = advanced_search("employee", {"document_type": "Policy"})
for i, doc in enumerate(policy_docs, 1):
    source = doc.metadata.get('source', 'Unknown')
    print(f"   {i}. {source}")

# Test 2: Search with department filter
print("\n2. Search 'benefits' with department='HR':")
hr_docs = advanced_search("benefits", {"department": "HR"})
for i, doc in enumerate(hr_docs, 1):
    source = doc.metadata.get('source', 'Unknown')
    print(f"   {i}. {source}")

# Test 3: Search with policy type filter
print("\n3. Search 'work' with policy_type='Remote':")
remote_docs = advanced_search("work", {"policy_type": "Remote"})
for i, doc in enumerate(remote_docs, 1):
    source = doc.metadata.get('source', 'Unknown')
    print(f"   {i}. {source}")

print("\n✅ Advanced search with filters completed")


In [ ]:
# Create advanced search tool for the agent
def advanced_search_tool(input_str: str) -> str:
    """Advanced search with filters and facets"""
    try:
        # Parse input (format: "query|filter1:value1|filter2:value2")
        parts = input_str.split('|')
        query = parts[0]
        filters = {}
        
        # Parse filters
        for part in parts[1:]:
            if ':' in part:
                key, value = part.split(':', 1)
                filters[key.strip()] = value.strip()
        
        # Perform advanced search
        results = advanced_search(query, filters)
        
        if not results:
            return f"No documents found for '{query}' with the specified filters."
        
        response = f"Advanced Search Results for '{query}':\n"
        if filters:
            response += f"Filters: {filters}\n"
        response += f"Found {len(results)} documents:\n\n"
        
        for i, doc in enumerate(results, 1):
            source = doc.metadata.get('source', 'Unknown')
            content_preview = doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content
            response += f"{i}. {source}\n   {content_preview}\n\n"
        
        return response
        
    except Exception as e:
        return f"Error in advanced search: {e}"

# Create faceted search tool
def faceted_search_tool(input_str: str) -> str:
    """Show available facets and search options"""
    try:
        response = "Available Search Facets:\n\n"
        
        for facet, values in document_metadata.items():
            response += f"📋 {facet.replace('_', ' ').title()}:\n"
            for value in values:
                response += f"   - {value}\n"
            response += "\n"
        
        response += "Usage: Use advanced_search_tool with format 'query|filter:value'\n"
        response += "Example: 'employee benefits|document_type:Policy|department:HR'\n"
        
        return response
        
    except Exception as e:
        return f"Error showing facets: {e}"

# Add tools to custom tools
advanced_search_tool_obj = Tool(
    name="advanced_search",
    description="Advanced search with filters. Input format: 'query|filter1:value1|filter2:value2'. Available filters: document_type, department, policy_type.",
    func=advanced_search_tool
)

faceted_search_tool_obj = Tool(
    name="faceted_search",
    description="Show available search facets and filters. Input: 'show_facets'.",
    func=faceted_search_tool
)

# Add to existing tools
custom_tools.append(advanced_search_tool_obj)
custom_tools.append(faceted_search_tool_obj)

print(f"✅ Added advanced search tools. Total tools: {len(custom_tools)}")


In [ ]:
# Test the advanced search tools
print("🧪 Testing Advanced Search Tools:")
print("=" * 50)

# Test 1: Show available facets
print("\n1. Available Search Facets:")
facets_result = faceted_search_tool("show_facets")
print(facets_result)

# Test 2: Advanced search with single filter
print("\n2. Advanced Search with Document Type Filter:")
advanced_result1 = advanced_search_tool("employee|document_type:Policy")
print(advanced_result1[:400] + "..." if len(advanced_result1) > 400 else advanced_result1)

# Test 3: Advanced search with multiple filters
print("\n3. Advanced Search with Multiple Filters:")
advanced_result2 = advanced_search_tool("benefits|department:HR|policy_type:Benefits")
print(advanced_result2[:400] + "..." if len(advanced_result2) > 400 else advanced_result2)

print("\n✅ Advanced search with filters and facets completed!")


### Advanced Search Features

**🔍 Semantic Search with Filters**
- **Document Type Filter**: Filter by Handbook, Policy, Benefits, Technical documents
- **Department Filter**: Filter by HR, IT, Finance departments
- **Policy Type Filter**: Filter by Leave Policy, Remote Work, Benefits, Safety
- **Combined Filters**: Use multiple filters together for precise results

**📊 Search Facets**
- **Document Types**: Handbook, Policy, Benefits, Technical
- **Departments**: HR, IT, Finance
- **Policy Types**: Leave Policy, Remote Work, Benefits, Safety
- **Dynamic Facets**: Automatically extracted from document content

**🛠️ Advanced Search Tools**
- **advanced_search**: Perform filtered searches with multiple criteria
- **faceted_search**: Show available search facets and filters
- **Smart Filtering**: Combines semantic search with metadata filtering
- **Flexible Input**: Supports single or multiple filter combinations

**💡 Business Value**
- **Precision**: Find exactly the documents you need with filters
- **Efficiency**: Narrow down results quickly without browsing
- **Discovery**: Explore documents by category and department
- **Compliance**: Find all related policies by type or department

**📝 Usage Examples**
- `employee|document_type:Policy` - Find policy documents about employees
- `benefits|department:HR|policy_type:Benefits` - Find HR benefits policies
- `work|policy_type:Remote` - Find remote work related documents

---


## 13. Summary and Complete Demo

### Enhanced Document Assistant Capabilities

The notebook now includes all advanced features:

✅ **LangChain Fundamentals**: Chat models, prompt templates, document processing
✅ **Vector Database & RAG**: ChromaDB with semantic search and retrieval  
✅ **Advanced Prompting**: Summarization and translation capabilities
✅ **Output Handling & Memory**: Conversation memory and structured responses
✅ **Tools & Agents**: Custom tools with intelligent agent orchestration
✅ **Document Relationships**: Automatic linking and relationship discovery
✅ **Advanced Search**: Semantic search with filters and facets

### New Demo Commands for Advanced Search

Try these advanced search queries with the agent:
- "Search for HR policy documents about benefits"
- "Find technical documents related to API"
- "Show me available search facets and filters"
- "Search employee|document_type:Policy|department:HR"

---


## 10. Summary and Next Steps

### What We've Built
This notebook implements a comprehensive **Intelligent Document Assistant** that combines:

✅ **LangChain Fundamentals**: Chat models, prompt templates, and document processing
✅ **Vector Database & RAG**: ChromaDB with semantic search and retrieval
✅ **Advanced Prompting**: Summarization and translation capabilities
✅ **Output Handling & Memory**: Conversation memory and structured responses
✅ **Tools & Agents**: Custom tools with intelligent agent orchestration
✅ **Document Relationships**: Automatic linking and relationship discovery

### Technical Architecture
```
Documents → Text Splitting → Embeddings → Vector DB
                ↓
User Query → RAG Pipeline → LLM → Response
                ↓
Custom Tools → Agent → Memory → Final Answer
```

### Demo Commands
Try these sample queries with the agent:
- "What is the vacation policy for new employees?"
- "Summarize the employee handbook for executives"
- "Translate 'company policies' to Spanish"
- "Find all documents about health insurance"
- "What are the requirements for promotion?"
- "Show me documents related to employee benefits"
- "Find relationships between vacation and benefits policies"

### Future Enhancements
1. **Multi-modal Support**: Handle images and tables in documents
2. **Real-time Updates**: Live document synchronization
3. **Advanced Analytics**: Document usage and query analytics
4. **Integration**: Connect with existing corporate systems
5. **Security**: Role-based access control and audit trails
6. **Knowledge Graphs**: Visual representation of document relationships
7. **Version Control**: Track document changes and relationship evolution

---

**🎉 Congratulations! You've successfully built an intelligent document assistant using LangChain!**
